### №4 **(3 балла)**
Выполните задания из файла https://cloud.mail.ru/public/nbue/PCozBiM4f. Данные можно
взять здесь https://cloud.mail.ru/public/ysqp/AuPY2gpwo.

### **УСЛОВИЕ**



В этом задании вам нужно проследить за изменением качества случайного леса в зависимости от количества деревьев в нем.

1. Загрузите данные из файла abalone.csv. Это датасет, в котором требуется предсказать возраст ракушки (число колец) по физическим
измерениям.

2. Преобразуйте признак Sex в числовой: значение F должно перейти
в -1, I — в 0, M — в 1. Если вы используете Pandas, то подойдет
следующий код: data[’Sex’] = data[’Sex’].map(lambda x: 1 if x ==
’M’ else (-1 if x == ’F’ else 0))

3. Разделите содержимое файлов на признаки и целевую переменную.
В последнем столбце записана целевая переменная, в остальных —
признаки.

4. Обучите случайный лес (sklearn.ensemble.RandomForestRegressor)
с различным числом деревьев: от 1 до 50 (random_state=1). Для
каждого из вариантов оцените качество работы полученного леса
на кросс-валидации по 5 блокам. Используйте параметры
"random_state=1"и "shuffle=True"при создании генератора кроссвалидации sklearn.cross_validation.KFold. В качестве меры качества
воспользуйтесь коэффициентом детерминации (sklearn.metrics.r2_score).

5. Определите, при каком минимальном количестве деревьев случайный лес показывает качество на кросс-валидации выше 0.52. Это
количество и будет ответом на задание.

6. Обратите внимание на изменение качества по мере роста числа деревьев. Ухудшается ли оно?

---

### **РЕШЕНИЕ**

In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

1. Загрузите данные из файла abalone.csv. Это датасет, в котором требуется предсказать возраст ракушки (число колец) по физическим
измерениям.

In [3]:
df = pd.read_csv('abalone_csv.csv', decimal=',')
df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Class_number_of_rings,Unnamed: 9,Unnamed: 10
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,NaN,NaN
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,NaN,NaN
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,NaN,NaN
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,NaN,NaN
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,NaN,NaN


In [4]:
df = df.drop(columns=['Unnamed: 9', 'Unnamed: 10'])
df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Class_number_of_rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [5]:
df.dtypes

Sex                       object
Length                   float64
Diameter                 float64
Height                   float64
Whole_weight             float64
Shucked_weight           float64
Viscera_weight           float64
Shell_weight             float64
Class_number_of_rings      int64
dtype: object

---

2. Преобразуйте признак Sex в числовой: значение F должно перейти
в -1, I — в 0, M — в 1.

Если вы используете Pandas, то подойдет
следующий код: data[’Sex’] = data[’Sex’].map(lambda x: 1 if x ==
’M’ else (-1 if x == ’F’ else 0))

In [8]:
df['Sex'] = df['Sex'].map(lambda x: 1 if x == 'M' else (-1 if x == 'F' else 0))
df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Class_number_of_rings
0,0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,0,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,0,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,0,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,0,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


---

3. Разделите содержимое файлов на признаки и целевую переменную.
В последнем столбце записана целевая переменная, в остальных —
признаки.

In [11]:
X = df.iloc[:, :-1]   # все столбцы, кроме последнего — признаки
y = df.iloc[:, -1]    # последний столбец — целевая переменная

---

4. Обучите случайный лес (sklearn.ensemble.RandomForestRegressor)
с различным числом деревьев: от 1 до 50 (random_state=1). Для
каждого из вариантов оцените качество работы полученного леса
на кросс-валидации по 5 блокам. Используйте параметры
"random_state=1"и "shuffle=True"при создании генератора кроссвалидации sklearn.cross_validation.KFold. В качестве меры качества
воспользуйтесь коэффициентом детерминации (sklearn.metrics.r2_score).

In [16]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)

n_trees = range(1, 51)
mean_scores = []

for n in n_trees:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    scores = cross_val_score(
        rf, X, y,
        cv=kf,
        scoring='r2'
    )
    mean_scores.append(scores.mean())
    print(f"n_estimators={n}: mean R²={scores.mean():.4f}")

best_idx = np.argmax(mean_scores)
best_n = n_trees[best_idx]
print("\nЛучшее число деревьев:", best_n)
print("Лучший средний R²:", mean_scores[best_idx])

n_estimators=1: mean R²=0.1230
n_estimators=2: mean R²=0.3319
n_estimators=3: mean R²=0.3996
n_estimators=4: mean R²=0.4393
n_estimators=5: mean R²=0.4588
n_estimators=6: mean R²=0.4665
n_estimators=7: mean R²=0.4753
n_estimators=8: mean R²=0.4800
n_estimators=9: mean R²=0.4867
n_estimators=10: mean R²=0.4905
n_estimators=11: mean R²=0.4941
n_estimators=12: mean R²=0.4975
n_estimators=13: mean R²=0.5020
n_estimators=14: mean R²=0.5057
n_estimators=15: mean R²=0.5076
n_estimators=16: mean R²=0.5095
n_estimators=17: mean R²=0.5120
n_estimators=18: mean R²=0.5126
n_estimators=19: mean R²=0.5141
n_estimators=20: mean R²=0.5127
n_estimators=21: mean R²=0.5140
n_estimators=22: mean R²=0.5155
n_estimators=23: mean R²=0.5161
n_estimators=24: mean R²=0.5177
n_estimators=25: mean R²=0.5172
n_estimators=26: mean R²=0.5185
n_estimators=27: mean R²=0.5192
n_estimators=28: mean R²=0.5204
n_estimators=29: mean R²=0.5219
n_estimators=30: mean R²=0.5221
n_estimators=31: mean R²=0.5222
n_estimators=32: 

---

5. Определите, при каком минимальном количестве деревьев случайный лес показывает качество на кросс-валидации выше 0.52. Это
количество и будет ответом на задание.

In [18]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)

n_trees = range(1, 51)
mean_scores = []

for n in n_trees:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    scores = cross_val_score(
        rf, X, y,
        cv=kf,
        scoring='r2'
    )
    mean_r2 = scores.mean()          # посчитали средний R²
    mean_scores.append(mean_r2)

    if mean_r2 > 0.52:               # строго «выше 0.52»
        print(f"минимальное n_estimators={n}: mean R²={mean_r2:.4f}")
        break

минимальное n_estimators=28: mean R²=0.5204


---

6. Обратите внимание на изменение качества по мере роста числа деревьев. Ухудшается ли оно?

**ОТВЕТ:** Качество в целом не ухудшается, а растёт 